# Practice 3 — Getting started with Hugging Face

**Môn Deep Learning — UTH.** Thực hiện đề giảng viên: (1) dùng mô hình sentiment pretrained, tokenize và suy luận; (2) tải dữ liệu nhị phân, tiền xử lý, cấu hình `Trainer`, fine-tune và đánh giá.

Notebook dùng chung mã nguồn với CLI. Mặc định **`MODE="view"`**: đọc và kiểm tra artifacts Kaggle có trong repo, không tải model/dataset và không train lại. Đổi thành **`"train"`** khi chủ động thực hiện thí nghiệm mới trên GPU. Các output lưu trong notebook này được thực thi ở chế độ view từ bằng chứng Kaggle; đây không phải lần GPU training mới trên máy tạo báo cáo.

Cài `labs/lab03/requirements.txt` trong Python 3.12 trước khi mở notebook. Hướng dẫn môi trường và sử dụng mô hình nằm trong `labs/lab03/README.md`.

In [1]:
from pathlib import Path
import sys
import json

LAB = next((p / "labs/lab03" if (p / "labs/lab03/src/workflow.py").is_file() else p
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "labs/lab03/src/workflow.py").is_file() or (p / "src/workflow.py").is_file()), None)
if LAB is None:
    raise RuntimeError("Open this notebook inside the Lab-DL repository")
sys.path.insert(0, str(LAB.parents[1]))

import pandas as pd
from IPython.display import display, Image, Markdown
from labs.lab03.src.workflow import read_config, exercise1, finetune
from labs.lab03.src.evidence import selected_run, audit_exercise1, audit_finetuning, read_json

# view: đọc và xác minh kết quả Kaggle đã lưu, không cần mạng/GPU.
# train: tạo thí nghiệm mới; Exercise 2 mặc định yêu cầu CUDA.
MODE = "view"
assert MODE in {"view", "train"}
config = read_config()
print("MODE:", MODE)
print("Lab directory:", LAB)
print("View outputs are from Kaggle artifacts, not a new training run.")

/workspace/lab03-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MODE: view
Lab directory: /workspace/Lab-DL/labs/lab03
View outputs are from Kaggle artifacts, not a new training run.


## Exercise 1 — Phân tích cảm xúc với mô hình pretrained

Mô hình `distilbert/distilbert-base-uncased-finetuned-sst-2-english` đã được fine-tune trên SST-2. Nhãn `NEGATIVE`/`POSITIVE` biểu diễn cảm xúc của câu. Confidence là xác suất softmax theo mô hình, không đảm bảo đúng.

`pipeline` tự thực hiện tokenize → forward → logits → softmax → argmax. Cách thủ công dùng `AutoTokenizer` và `AutoModelForSequenceClassification`; code kiểm tra hai cách cho cùng nhãn và score lệch không quá `1e-5`.

In [2]:
exercise1_run = exercise1(config) if MODE == "train" else selected_run("exercise1")
exercise1_audit = audit_exercise1(exercise1_run)
print(json.dumps(exercise1_audit, indent=2))
print("Recorded environment:")
display(read_json(exercise1_run / "environment_snapshot.json"))

{
  "verified_files": 6,
  "omitted_checkpoint_files": [],
  "checkpoint_available_locally": false,
  "run": "20261025T181532Z_exercise1_71da6f40",
  "compared_sentences": 3
}
Recorded environment:


{'python': '3.12.15 (main, Oct 21 2026, 17:47:06) [Clang 23.1.3 ]',
 'platform': 'Linux-6.18.55+-x86_64-with-glibc2.39',
 'cuda_available': True,
 'device': 'Tesla T4',
 'versions': {'torch': '2.6.0',
  'transformers': '4.48.3',
  'datasets': '3.2.0',
  'evaluate': '0.4.3',
  'accelerate': '1.3.0',
  'numpy': '2.2.2',
  'scikit-learn': '1.6.1'}}

### Tokenization — WordPiece và attention mask

`input_ids` ánh xạ token sang vocabulary ID. `[CLS]` bắt đầu chuỗi, `[SEP]` kết thúc. Mask `1` là token được chú ý, `0` là padding. Với câu đơn chưa padding, mask toàn `1` là đúng. Từng token trong bảng dưới được đọc từ artifact của run đã xác minh.

In [3]:
breakdown = read_json(exercise1_run / "tokenizer_breakdown.json")
print("Sentence:", breakdown["text"])
display(pd.DataFrame({k: breakdown[k] for k in ["tokens", "input_ids", "attention_mask"]}))

Sentence: I love this wonderful movie!


,tokens,input_ids,attention_mask
0,[CLS],101,1
1,i,1045,1
2,love,2293,1
3,this,2023,1
4,wonderful,6919,1
5,movie,3185,1
6,!,999,1
7,[SEP],102,1


### Đối chiếu pipeline và forward thủ công

Với logits $z$, xác suất $p_c = e^{z_c}/\sum_j e^{z_j}$. Chọn lớp có xác suất lớn nhất. Mô hình ở `eval()` và dùng `torch.inference_mode()` khi suy luận; không cập nhật trọng số trong Exercise 1.

In [4]:
predictions = read_json(exercise1_run / "predictions.json")
comparison = []
for high, low in zip(predictions["pipeline"], predictions["manual"]):
    comparison.append({"text": low["text"], "pipeline_label": high["label"],
        "manual_label": low["label"], "pipeline_score": high["score"],
        "manual_score": low["score"], "absolute_difference": abs(high["score"]-low["score"])})
display(pd.DataFrame(comparison))

,text,pipeline_label,manual_label,pipeline_score,manual_score,absolute_difference
0,I love this wonderful movie!,POSITIVE,POSITIVE,0.999878,0.999878,0.0
1,This movie was terrible and boring.,NEGATIVE,NEGATIVE,0.999768,0.999768,0.0
2,"The acting was good, but the story was disappo...",NEGATIVE,NEGATIVE,0.996749,0.996749,0.0


### Nhận xét

Câu “I love this wonderful movie!” được gán tích cực; câu “terrible and boring” được gán tiêu cực. Câu “The acting was good, but the story was disappointing.” có hai sắc thái, mô hình chọn tiêu cực. Đối chiếu ba câu xác nhận hai cách suy luận nhất quán; ba ví dụ không đo được accuracy tổng quát của mô hình.